# Lab 1: Profiling a model and measuring your own roofline
**AIMS Fast ML · Day 2 · ~2 hours**

On Day 1 we said a model is only useful if it fits its **budget**: latency, throughput, power and memory.
Today you measure where a small model actually sits on **your own laptop**, and why.

By the end of this lab you will have:
1. Counted a network's parameters, operations and bytes **by hand**, and turned them into an *arithmetic intensity*.
2. Timed inference properly: warm-up, percentiles, threads, batching.
3. Measured your machine's peak compute and memory bandwidth, and drawn **your own roofline**.
4. Placed the model on that roofline and explained what limits it.
5. Compared all of this to the LHC Level-1 trigger budget.

| Mark | Meaning |
|---|---|
| 🟥 **AI-off** | Pen and paper. No AI tools. Questions like these will be on the exam. |
| 🟩 **AI-on** | Use whatever tools you like. You must run the code on *your* machine and be able to explain it. |

> **Before you start:** plug your laptop in and close heavy apps (browsers with many tabs, video calls).
> Laptops throttle on battery and under heat. Write down which state you measured in.

## 0. Setup (5 min)

In [ ]:
import sys, os, time, platform
sys.path.append("..")                      # makes `common/` importable
import numpy as np, torch, torch.nn as nn
import matplotlib.pyplot as plt
from common.jets import load_jets, CLASSES

torch.manual_seed(0); np.random.seed(0)
N_CORES = os.cpu_count()
print(f"Python {platform.python_version()} | torch {torch.__version__} | numpy {np.__version__}")
print(f"{platform.system()} {platform.machine()} | {platform.processor() or '?'} | {N_CORES} logical cores")
print(f"torch threads: {torch.get_num_threads()}")

Note your CPU model; you will need it in Section 5. On macOS run `sysctl -n machdep.cpu.brand_string` in a terminal,
on Linux `lscpu`, on Windows open *Task Manager → Performance*.

*My CPU:* ______________________  *Plugged in?* ____

## 1. The model (10 min)
We use the standard fast-ML benchmark: **jet tagging** at the LHC. Each jet is described by 16 high-level features,
and the task is to classify it as coming from a gluon, a light quark, a W, a Z or a top quark.
The model is the small MLP used throughout the hls4ml literature: **16 → 64 → 32 → 32 → 5**, ReLU.

In [ ]:
X_train, X_test, y_train, y_test, features = load_jets()
print(X_train.shape, X_test.shape, "classes:", CLASSES)

class JetMLP(nn.Module):
    def __init__(self, sizes=(16, 64, 32, 32, 5)):
        super().__init__()
        layers = []
        for i, (a, b) in enumerate(zip(sizes[:-1], sizes[1:])):
            layers.append(nn.Linear(a, b))
            if i < len(sizes) - 2:
                layers.append(nn.ReLU())
        self.net = nn.Sequential(*layers)
    def forward(self, x):
        return self.net(x)

model = JetMLP()
print(model)

In [ ]:
# Quick training. Takes ~30-60 s on a laptop CPU. Today is not about accuracy, but we want a real model.
def train(model, X, y, epochs=3, bs=1024, lr=3e-3):
    Xt, yt = torch.from_numpy(X), torch.from_numpy(y)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    lossf = nn.CrossEntropyLoss()
    for ep in range(epochs):
        perm = torch.randperm(len(Xt))
        for i in range(0, len(Xt), bs):
            idx = perm[i:i + bs]
            opt.zero_grad(); loss = lossf(model(Xt[idx]), yt[idx]); loss.backward(); opt.step()
        print(f"epoch {ep}: loss {loss.item():.3f}")
    return model

t0 = time.perf_counter(); train(model, X_train, y_train); print(f"trained in {time.perf_counter()-t0:.0f} s")
model.eval()
with torch.inference_mode():
    acc = (model(torch.from_numpy(X_test)).argmax(1).numpy() == y_test).mean()
print(f"test accuracy: {acc:.3f}   (the literature value for this model is ~0.75)")

## 2. 🟥 AI-off: the budget on paper (15 min)
Answer these **on paper first**, then check them with code.
Conventions: one multiply-accumulate (MAC) = 2 FLOPs. Ignore biases and ReLUs in the FLOP count, but **include** biases in the parameter count. Use fp32 (4 bytes).

**Q2.1** How many parameters does the model have?

**Q2.2** How many MACs (and FLOPs) does one inference (batch size 1) take?

**Q2.3** How many bytes must be read and written for one inference? Assume the weights come from main memory every call,
and count input and output. Assume intermediate activations stay in cache.

**Q2.4** The *arithmetic intensity* is $I = \text{FLOPs} / \text{bytes}$. Compute $I$ for batch size 1.
Then write $I(B)$ for batch size $B$, assuming the weights are read **once per batch**. What is $\lim_{B\to\infty} I(B)$,
and what sets that limit?

*Your answers:*


In [ ]:
def count_params(model):
    return sum(p.numel() for p in model.parameters())

def count_macs(model):
    """MACs for one sample through every nn.Linear layer (ignore biases and activations)."""
    # YOUR CODE HERE
    raise NotImplementedError

def bytes_moved(model, batch, dtype_bytes=4):
    """Bytes for one batch: weights read once, plus input and output of every sample."""
    lin = [m for m in model.modules() if isinstance(m, nn.Linear)]
    # YOUR CODE HERE
    raise NotImplementedError

def intensity(model, batch):
    return 2 * count_macs(model) * batch / bytes_moved(model, batch)

print("params", count_params(model), "| MACs", count_macs(model), "| bytes(B=1)", bytes_moved(model, 1))
for B in [1, 16, 256, 4096, 10**6]:
    print(f"I(B={B:>7}) = {intensity(model, B):6.2f} FLOP/byte")
assert count_macs(model) == 4256 and bytes_moved(model, 1) == 17640

## 3. 🟩 Timing inference properly (25 min)
Timing code looks easy and is easy to get wrong. Start with the naive approach:

In [ ]:
x1 = torch.from_numpy(X_test[:1])
for _ in range(5):
    t0 = time.time(); model(x1); print(f"{(time.time()-t0)*1e6:8.1f} µs")

**Q3.1** Why is the first call slower? What does `time.time()` resolution do to measurements this short?

Write a proper timing helper. It should:
- run `n_warmup` untimed calls,
- time each of `n_iter` calls with `time.perf_counter_ns()`,
- return an array of per-call times **in microseconds**.

In [ ]:
def measure(fn, n_warmup=50, n_iter=500):
    # YOUR CODE HERE
    raise NotImplementedError

def summary(t, label=""):
    print(f"{label:<28} median {np.median(t):8.2f} µs | p99 {np.percentile(t, 99):8.2f} µs | min {t.min():8.2f} µs")

t_grad = measure(lambda: model(x1))
with torch.inference_mode():
    t_inf = measure(lambda: model(x1))
summary(t_grad, "batch 1, autograd on"); summary(t_inf, "batch 1, inference_mode")

plt.hist(t_inf, bins=80, log=True); plt.xlabel("latency [µs]"); plt.ylabel("count"); plt.title("batch-1 latency distribution"); plt.show()

**Q3.2** Look at the histogram. Why do we quote **p99** (or p99.9) and not just the mean? Think of a trigger that must
decide on *every* event.

Now compare against the same network written directly in NumPy, with no framework:

In [ ]:
W = [(m.weight.detach().numpy().T.copy(), m.bias.detach().numpy().copy()) for m in model.modules() if isinstance(m, nn.Linear)]

def numpy_forward(x):
    for i, (w, b) in enumerate(W):
        x = x @ w + b
        if i < len(W) - 1:
            x = np.maximum(x, 0)
    return x

x1_np = X_test[:1]
assert np.allclose(numpy_forward(x1_np), model(x1).detach().numpy(), atol=1e-5)
summary(measure(lambda: numpy_forward(x1_np)), "batch 1, numpy")

for n in [1, N_CORES]:
    torch.set_num_threads(n)
    with torch.inference_mode():
        summary(measure(lambda: model(x1)), f"batch 1, torch, {n} thread(s)")
torch.set_num_threads(N_CORES)

**Q3.3** The model does about 8.5k FLOPs, so a 1 GFLOP/s core would need ~8.5 µs and a modern core well under 1 µs. How does that compare
to what you measured? Where is the rest of the time going? Why do more threads not help (and may hurt) at batch 1?



## 4. 🟩 Latency vs. throughput: the batch sweep (20 min)
Sweep the batch size, and record the latency per *batch* and the throughput in *samples per second*.

In [ ]:
batches = [1, 4, 16, 64, 256, 1024, 4096, 16384, 65536]
Xbig = torch.from_numpy(np.resize(X_test, (max(batches), 16)).astype(np.float32))
lat, thr = [], []
with torch.inference_mode():
    for B in batches:
        xb = Xbig[:B]
        n_iter = max(5, min(300, int(2e6 / B)))   # keep each point to a fraction of a second
        t = measure(lambda: model(xb), n_warmup=5, n_iter=n_iter)
        lat.append(np.median(t)); thr.append(B / (np.median(t) * 1e-6))
        print(f"B={B:>6}: {lat[-1]:10.1f} µs/batch  {thr[-1]/1e6:8.2f} M samples/s")

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].loglog(batches, lat, "o-"); ax[0].set(xlabel="batch size", ylabel="latency per batch [µs]")
ax[1].semilogx(batches, np.array(thr) / 1e6, "o-"); ax[1].set(xlabel="batch size", ylabel="throughput [M samples/s]")
for a in ax: a.grid(alpha=.3)
plt.tight_layout(); plt.show()

**Q4.1** Identify the regimes in the left plot. Over what range is latency roughly *flat* in $B$? What does that tell you
about the cost of a batch-1 call?

**Q4.2** 🟥 *AI-off.* **Little's law:** (samples in flight) = throughput × latency. A radio-telescope pipeline needs 50 M samples/s
and can tolerate 10 ms latency. A trigger needs 40 M samples/s and **1 µs** latency. Using your plots, which can your laptop serve, and how?



## 5. 🟩 Your own roofline (25 min)
The **roofline model**: attainable FLOP/s $= \min(P_\text{peak},\; I \times \beta)$, where $\beta$ is the memory bandwidth.
Now measure $P_\text{peak}$ and $\beta$ on your machine.

**5a. Peak compute.** A large square matmul has a high arithmetic intensity, so it runs close to peak.

In [ ]:
def matmul_gflops(n=2048, reps=5):
    a, b = torch.randn(n, n), torch.randn(n, n)
    # YOUR CODE HERE
    raise NotImplementedError

for n in [256, 1024, 2048]:
    print(f"matmul n={n}: {matmul_gflops(n):8.1f} GFLOP/s")
P_peak = matmul_gflops(2048) * 1e9

**5b. Memory bandwidth.** Use a STREAM-like *triad* `a = b + s*c` on arrays far bigger than any cache.
Count the bytes carefully: two reads and one write per element.

In [ ]:
def triad_gbs(n=32 * 2**20, reps=5):
    b, c = torch.randn(n), torch.randn(n); a = torch.empty(n)
    # YOUR CODE HERE
    raise NotImplementedError

for n in [2**14, 2**17, 2**20, 32 * 2**20]:
    print(f"triad n={n:>10} ({3*4*n/2**20:7.1f} MiB touched): {triad_gbs(n):8.1f} GB/s")
beta = triad_gbs() * 1e9
print(f"\nP_peak ≈ {P_peak/1e9:.0f} GFLOP/s,  β ≈ {beta/1e9:.0f} GB/s,  ridge point ≈ {P_peak/beta:.1f} FLOP/byte")

**Q5.1** The mid-size arrays probably give a much *higher* "bandwidth" than the largest, and the smallest probably give a *lower* one.
Explain both. What are you really measuring in each case? Which number is the DRAM bandwidth $\beta$?


Now draw the roofline and place the jet MLP on it at each batch size, using the intensity from Section 2 and the throughput from Section 4.

In [ ]:
I_axis = np.logspace(-2, 3, 200)
B_arr = np.array(batches)
I_mlp = np.array([intensity(model, B) for B in batches])
F_mlp = np.array(thr) * 2 * count_macs(model)           # achieved FLOP/s

plt.figure(figsize=(7, 5))
plt.loglog(I_axis, np.minimum(P_peak, I_axis * beta) / 1e9, "k-", lw=2, label="roofline (measured)")
sc = plt.scatter(I_mlp, F_mlp / 1e9, c=np.log2(B_arr), cmap="viridis", zorder=3, label="jet MLP")
for B, i, f in zip(batches, I_mlp, F_mlp):
    plt.annotate(f"B={B}", (i, f / 1e9), textcoords="offset points", xytext=(5, -10), fontsize=8)
plt.axvline(P_peak / beta, ls=":", c="gray"); plt.text(P_peak / beta * 1.1, P_peak / 1e9 * 0.05, "ridge", color="gray")
plt.xlabel("arithmetic intensity [FLOP/byte]"); plt.ylabel("performance [GFLOP/s]")
plt.title("My laptop's roofline"); plt.legend(loc="lower right"); plt.grid(alpha=.3, which="both"); plt.show()

**Q5.2** At B=1 the model sits **far** below the roof, often 100× or more. Is it compute-bound, memory-bound, or neither?
Connect this to Q3.3.

**Q5.3** At large B, how close do you get to the roof? The weights (17.6 kB) live in cache, so is DRAM bandwidth even the right
roof for this model? Which roof would be? *(Hint: look at your small-`n` triad numbers.)*

**Q5.4** 🟥 *AI-off.* In one paragraph: what would you change (in software or hardware) to move the B=1 point up? We spend
Days 3–8 on exactly this.



## 6. 🟥 AI-off: back to the trigger budget (10 min)
The CMS Level-1 trigger sees a bunch crossing every **25 ns** (40 MHz). A neural network running there might get on the order of
**100 ns** of latency and must accept a new input every 25 ns.

**Q6.1** What sustained FLOP/s does running this MLP on every crossing need? Is it above or below your measured $P_\text{peak}$?

**Q6.2** So compute isn't the problem. Give **three** distinct reasons why your laptop still cannot do this job.

**Q6.3** An FPGA at 200 MHz performs all 4256 MACs of this network *in parallel*, one multiplier per MAC. How many clock cycles
does 100 ns give you, and what does that say about how deep the pipeline can be?



## 7. Wrap-up
Save your two key plots (the batch sweep and your roofline), and record your $P_\text{peak}$, $\beta$, and batch-1 p99 latency.
**Assignment A1** builds on these exact numbers.

| My machine | |
|---|---|
| CPU | |
| $P_\text{peak}$ [GFLOP/s] | |
| $\beta$ [GB/s] | |
| ridge point [FLOP/B] | |
| batch-1 median / p99 latency [µs] | |
| best throughput [M samples/s] at B = | |